# Fine-tune WangchanBERTa สำหรับ Auto Engagement API

Notebook นี้ fine-tune [WangchanBERTa](https://huggingface.co/airesearch/wangchanberta-base-att-spm-uncased)
เป็นตัวจำแนก **sentiment** และ **category** แล้ว export เป็น ONNX (int8) ให้ API ใช้งานบน CPU

**ก่อนเริ่ม:** เลือก Runtime → Change runtime type → **T4 GPU**
ใช้เวลาประมาณ 15–25 นาทีต่อโมเดลบน T4

ใช้ข้อมูลและ train/test split ชุดเดียวกับ TF-IDF+SVM (`data/*_dataset_v2.csv`) ผลจึงเทียบกันได้ตรงๆ

In [ ]:
# STEP 0: เช็คว่าได้ GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# STEP 1: ดึงโค้ดและติดตั้ง library
!git clone -q https://github.com/67130043-tech/auto-engagement-api-new-version.git
%cd auto-engagement-api-new-version
!pip install -q -r requirements.txt -r requirements-train.txt

## (ไม่บังคับ) อัปโหลดโมเดลขึ้น HuggingFace Hub
โมเดลมีขนาด ~110 MB เกินขีดจำกัด 100 MB ต่อไฟล์ของ GitHub วิธีที่แนะนำคือเก็บบน HuggingFace Hub
แล้วตั้งค่า `WANGCHAN_SENTIMENT_REPO` / `WANGCHAN_CATEGORY_REPO` บน Render ให้ API ดาวน์โหลดเอง

1. สมัคร/ล็อกอิน https://huggingface.co แล้วสร้าง Access Token แบบ **Write**
2. รัน cell ด้านล่าง วาง token แล้วแก้ `HF_USER` เป็นชื่อบัญชีของคุณ

ถ้าข้ามขั้นนี้ ให้ใช้ STEP 4 ดาวน์โหลดไฟล์ zip แทน

In [ ]:
from huggingface_hub import notebook_login
notebook_login()
HF_USER = "your-hf-username"   # <- แก้เป็นชื่อบัญชี HuggingFace ของคุณ
PUSH = f"--push-to-hub {HF_USER}/wangchanberta-autoengage"

In [ ]:
# STEP 2: Fine-tune sentiment (Wisesight + Wongnai)
push = f"{PUSH}-sentiment" if "PUSH" in globals() else ""
!python 11_finetune_wangchanberta.py --task sentiment --epochs 3 {push}

In [ ]:
# STEP 3: Fine-tune category (10 หมวด)
push = f"{PUSH}-category" if "PUSH" in globals() else ""
!python 11_finetune_wangchanberta.py --task category --epochs 4 {push}

In [ ]:
# STEP 4: เทียบผลกับ TF-IDF+SVM บน test set เดียวกัน และดาวน์โหลดผล
!python 02_train_models.py > /dev/null
!python 12_compare_models.py
!zip -qr wangchanberta_models.zip models/wangchanberta_sentiment models/wangchanberta_category outputs/model_comparison.csv outputs/model_comparison.json
from google.colab import files
files.download("wangchanberta_models.zip")

## นำไปใช้กับ API
- **ทางที่แนะนำ:** ตั้ง Environment Variable บน Render
  - `WANGCHAN_SENTIMENT_REPO = <HF_USER>/wangchanberta-autoengage-sentiment`
  - `WANGCHAN_CATEGORY_REPO = <HF_USER>/wangchanberta-autoengage-category`
  - ถ้า repo เป็น private ให้ตั้ง `HF_TOKEN` ด้วย
- **หรือ** แตก zip แล้ววางโฟลเดอร์ `models/wangchanberta_sentiment/` และ `models/wangchanberta_category/` ไว้บนเครื่องที่รัน API

เมื่อพบโมเดล API จะใช้ WangchanBERTa ตัดสิน sentiment/category เอง (keyword เหลือแค่กฎความปลอดภัย)
ถ้าไม่พบ จะใช้ TF-IDF+SVM แบบเดิมอัตโนมัติ — ดู `/` endpoint ว่ากำลังใช้โมเดลไหน